# This Notebook will build a Machine Learning model on Supplier Late Payment and predict the Supplier Risk Score

In [1]:
# Define Parameters
silver_catalog="demo_fusion_silver"
silver_schema="demo_fusion_silver_schema"
gold_catalog      = "demo_fusion_gold"
gold_schema       = "aidp"

In [2]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

In [4]:
#Index categorical columns
supplier_indexer = StringIndexer(inputCol="supplier_id", outputCol="SUPPLIER_IDX", handleInvalid="keep")
org_indexer = StringIndexer(inputCol="apinvoicesorgid", outputCol="ORG_IDX", handleInvalid="keep")
currency_indexer = StringIndexer(inputCol="apinvoicesinvoicecurrencycode", outputCol="CURRENCY_IDX", handleInvalid="keep")


In [5]:
fusion_invoice_supplier_df=spark.read.table(f"{gold_catalog}.{gold_schema}.fusion_invoice_supplier")
fusion_invoice_supplier_df.printSchema()

root
 |-- apinvoicesinvoiceid: string (nullable = true)
 |-- apinvoicesinvoicedate: timestamp (nullable = true)
 |-- appaymenthistorypaymentdate: timestamp (nullable = true)
 |-- term_days: decimal(5,0) (nullable = true)
 |-- due_date: timestamp (nullable = true)
 |-- days_to_pay: decimal(10,2) (nullable = true)
 |-- payment_delay: decimal(10,2) (nullable = true)
 |-- late_payment_flag: decimal(1,0) (nullable = true)
 |-- apinvoicesvendorid: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- apinvoicesinvoiceamount: decimal(18,2) (nullable = true)
 |-- apinvoicesbaseamount: decimal(18,2) (nullable = true)
 |-- apinvoicesinvoicecurrencycode: string (nullable = true)
 |-- apinvoicespaymentstatusflag: string (nullable = true)
 |-- apinvoicesapprovalstatus: string (nullable = true)
 |-- apinvoicesorgid: string (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-

In [6]:
fact_df_indexed = supplier_indexer.fit(fusion_invoice_supplier_df).transform(fusion_invoice_supplier_df)
fact_df_indexed = org_indexer.fit(fact_df_indexed).transform(fact_df_indexed)
fact_df_indexed = currency_indexer.fit(fact_df_indexed).transform(fact_df_indexed)

In [7]:
# Assemble features
feature_cols = ["apinvoicesinvoiceamount", "term_days", "SUPPLIER_IDX", "ORG_IDX", "CURRENCY_IDX"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

ml_df = assembler.transform(fact_df_indexed).select("features", "late_payment_flag", "supplier_id", "apinvoicesinvoiceid")

In [8]:
# Split train/test
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)


In [9]:
#Train Random Forest classifier
rf = RandomForestClassifier(labelCol="late_payment_flag", featuresCol="features", numTrees=50, maxDepth=5)
rf_model = rf.fit(train_df)

# Train Machine Learning model

In [10]:
# Make predictions
predictions = rf_model.transform(test_df)

In [11]:
#Evaluate
evaluator = BinaryClassificationEvaluator(labelCol="late_payment_flag", metricName="areaUnderROC")
roc_auc = evaluator.evaluate(predictions)
print(f"ROC-AUC: {roc_auc}")

ROC-AUC: 0.6319586914205815


In [12]:
#Generate supplier-level risk score

predictions_full = rf_model.transform(train_df)
from pyspark.sql.functions import col, avg
supplier_risk_score_df = predictions_full.groupBy("supplier_id").agg(
    avg(col("prediction")).alias("SUPPLIER_RISK_SCORE")
)

supplier_risk_score_df.show()

+---------------+--------------------+
|    supplier_id| SUPPLIER_RISK_SCORE|
+---------------+--------------------+
|300000047414571|                 0.0|
|300000047414503|  0.0215318869165023|
|           NULL| 0.01611928873148507|
|300000047572113|                 0.0|
|300000047414679|0.023561432609863795|
|300000047414635| 0.02944880881222782|
|300000047507596|                 0.0|
+---------------+--------------------+



# Write Supplier Score Predictions to Gold Catalog

In [13]:
supplier_risk_score_df.write.insertInto(f"{gold_catalog}.{gold_schema}.FUSION_SUPPLIER_RISK_SCORE")